In [1]:
#Sample log file
import random
from pathlib import Path
from datetime import datetime, timedelta

LOG_FILE = Path("server.log")
TOTAL_LINES = 200_000

random.seed(42)
LEVELS  = ["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL", "FATAL"]
WEIGHTS = [20, 60, 10, 7, 2, 1]

ERROR_TYPES = {
    "DatabaseTimeout":      "Connection to db-primary timed out",
    "NullPointerException": "Object reference is null in OrderService",
    "OutOfMemoryError":     "Java heap space exhausted",
    "AuthFailure":          "Invalid token for user session",
    "DiskFull":             "No space left on /var/data",
    "NetworkUnreachable":   "Upstream payment gateway unreachable",
}
NORMAL_TYPES = {"DEBUG": "CacheLookup", "INFO": "RequestHandled", "WARNING": "SlowQuery"}

ts = datetime(2026, 10, 23, 0, 0, 0)

with open(LOG_FILE, "w", encoding="utf-8") as f:
    for _ in range(TOTAL_LINES):
        ts += timedelta(milliseconds=400)
        level = random.choices(LEVELS, WEIGHTS)[0]

        if level in NORMAL_TYPES:
            etype, msg = NORMAL_TYPES[level], "operation completed"
        else:
            etype = random.choice(list(ERROR_TYPES))
            msg = ERROR_TYPES[etype]

        f.write(f"{ts:%Y-%m-%d %H:%M:%S} | {level} | {etype} | {msg}\n")

        r = random.random()
        if r < 0.002:
            f.write("    at com.shop.OrderService.process(OrderService.java:88)\n")  # stack trace
        elif r < 0.003:
            f.write("@@@ garbage line ###\n")                                       # kharab line

size_mb = LOG_FILE.stat().st_size / 1_048_576
print(f"✅ {LOG_FILE} ban gayi: {size_mb:.1f} MB")

✅ server.log ban gayi: 12.8 MB


In [2]:
#Streaming generators
import re

LINE_RE = re.compile(
    r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s*\|\s*(\w+)\s*\|\s*([\w.]+)\s*\|\s*(.*)$"
)
CRITICAL_LEVELS = {"ERROR", "CRITICAL", "FATAL"}


def read_lines(path):
    """File ek line ek waqt par yield karo. Kharab bytes replace ho jaate hain, crash nahi."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:                      # lazy: poori file load nahi hoti
            yield line.rstrip("\n")


def critical_errors(path, stats):
    """Sirf critical entries yield karo."""
    for line in read_lines(path):
        stats["scanned"] += 1
        match = LINE_RE.match(line)
        if not match:
            stats["skipped"] += 1
            continue
        ts, level, etype, msg = match.groups()
        level = level.upper()
        if level in CRITICAL_LEVELS:
            yield {"time": ts, "level": level, "type": etype, "message": msg}

In [3]:
#Live monitor (grouping + running count)
import sys, time, tracemalloc
from collections import Counter, defaultdict, deque


def new_group():
    return {"count": 0, "first": None, "last": None, "samples": deque(maxlen=3)}


def live_monitor(path, refresh=0.1, top_n=3, demo_delay=0.0):
    stats = {"scanned": 0, "skipped": 0}
    groups = defaultdict(new_group)
    counts = Counter()
    total = 0
    last_draw = 0.0

    def draw():
        top = " | ".join(f"{t}: {c:,}" for t, c in counts.most_common(top_n))
        line = f"Scanned: {stats['scanned']:,} | Critical: {total:,} | {top}"
        sys.stdout.write("\r" + line.ljust(120))
        sys.stdout.flush()

    try:
        for entry in critical_errors(path, stats):
            total += 1
            g = groups[entry["type"]]
            g["count"] += 1
            g["first"] = g["first"] or entry["time"]
            g["last"] = entry["time"]
            g["samples"].append(entry)
            counts[entry["type"]] += 1

            if demo_delay:
                time.sleep(demo_delay)       # sirf demo ke liye, slow-motion
            now = time.monotonic()
            if now - last_draw >= refresh:
                draw()
                last_draw = now
        draw()                               # aakhri update
        print()
    except FileNotFoundError:
        print(f"❌ File nahi mili: {path}")
    except PermissionError:
        print(f"❌ File kholne ki permission nahi: {path}")
    except KeyboardInterrupt:
        print("\n⏹️ Rok diya gaya, ab tak ka result neeche hai.")

    return total, groups, stats


tracemalloc.start()
t0 = time.perf_counter()
total, groups, stats = live_monitor(LOG_FILE)
elapsed = time.perf_counter() - t0
_, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

Scanned: 200,582 | Critical: 19,985 | OutOfMemoryError: 3,357 | DatabaseTimeout: 3,354 | NetworkUnreachable: 3,335      


In [4]:
#Final summary
LINE = "=" * 68
print(LINE)
print("SERVER LOG ERROR SUMMARY".center(68))
print(LINE)
print(f"Lines scanned : {stats['scanned']:,}  (skipped/malformed: {stats['skipped']:,})")
print(f"Critical total: {total:,}")
print(f"Time taken    : {elapsed:.2f}s")
print(f"File size     : {LOG_FILE.stat().st_size / 1_048_576:.1f} MB "
      f"| Peak memory: {peak / 1024:.0f} KB")

print("\nBY ERROR TYPE")
print("-" * 68)
for etype, g in sorted(groups.items(), key=lambda kv: kv[1]["count"], reverse=True):
    pct = g["count"] / total * 100 if total else 0
    bar = "█" * int(pct / 2)
    print(f"{etype:<22}{g['count']:>7,}  {pct:5.1f}%  {bar}")
    print(f"    first: {g['first']}   last: {g['last']}")
    print(f"    latest: [{g['samples'][-1]['level']}] {g['samples'][-1]['message']}")
print(LINE)

                      SERVER LOG ERROR SUMMARY                      
Lines scanned : 200,582  (skipped/malformed: 582)
Critical total: 19,985
Time taken    : 4.18s
File size     : 12.8 MB | Peak memory: 56 KB

BY ERROR TYPE
--------------------------------------------------------------------
OutOfMemoryError        3,357   16.8%  ████████
    first: 2026-10-23 00:00:05   last: 2026-10-23 22:12:28
    latest: [ERROR] Java heap space exhausted
DatabaseTimeout         3,354   16.8%  ████████
    first: 2026-10-23 00:00:24   last: 2026-10-23 22:12:58
    latest: [CRITICAL] Connection to db-primary timed out
NetworkUnreachable      3,335   16.7%  ████████
    first: 2026-10-23 00:00:12   last: 2026-10-23 22:12:54
    latest: [ERROR] Upstream payment gateway unreachable
AuthFailure             3,329   16.7%  ████████
    first: 2026-10-23 00:00:15   last: 2026-10-23 22:12:39
    latest: [CRITICAL] Invalid token for user session
NullPointerException    3,312   16.6%  ████████
    first: 2026-